# Are the gaps bigger than chance? (Phase 4: statistics)

Phase 3 found gaps between groups. Here each one gets two checks: **is it bigger than chance
would explain**, and **how big is it** (with a 95% confidence interval). These are still
associations in past orders, not causes. Numbers come from the **analytics** tables built by
`python -m src.pipeline`.

**Contents:** 1. Setup · 2. How sure are we of one rate?

## 1. Setup
Same connection helper as the first notebook. **SciPy** is Python's standard library for
statistics; we use its tests and intervals instead of writing the formulas ourselves.

In [1]:
import math
import sys
from pathlib import Path

import pandas as pd
from scipy import stats
from sqlalchemy import text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))            # so "from src..." works when run from notebooks/
from src.db import get_engine

engine = get_engine()

def query(sql: str) -> pd.DataFrame:
    """Run one SQL query and return the result as a table."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

## 2. How sure are we of one rate?
The main KPI is the unhappy rate: 1-2 star orders ÷ reviewed orders. Our orders are one sample of
what Olist's customers do, so the rate would wobble a little with a different sample. A **95%
confidence interval** is the range the true rate probably lies in: if we repeated the sampling
many times, 95% of such ranges would contain it.

We use the **Wilson** interval (SciPy's `binomtest(...).proportion_ci`), which stays accurate for
small groups and rates near 0% or 100%. The simple textbook formula, rate ± 1.96 × standard error,
is shown next to it as a check.

In [2]:
def rate_ci(unhappy: int, reviewed: int) -> dict:
    """Unhappy rate with a 95% Wilson interval, plus the simple formula as a check (all in %)."""
    ci = stats.binomtest(unhappy, reviewed).proportion_ci(confidence_level=0.95, method="wilson")
    p = unhappy / reviewed
    se = math.sqrt(p * (1 - p) / reviewed)          # standard error: typical wobble of the rate
    return {"reviewed": reviewed, "unhappy": unhappy, "rate_pct": round(100 * p, 2),
            "wilson_low": round(100 * ci.low, 2), "wilson_high": round(100 * ci.high, 2),
            "simple_low": round(100 * (p - 1.96 * se), 2), "simple_high": round(100 * (p + 1.96 * se), 2)}

kpi = query("""
    SELECT count(*) FILTER (WHERE has_review) AS reviewed, count(*) FILTER (WHERE is_unhappy) AS unhappy
    FROM analytics.fact_order
""").iloc[0]
pd.DataFrame([rate_ci(int(kpi["unhappy"]), int(kpi["reviewed"]))], index=["all reviewed orders"])

,reviewed,unhappy,rate_pct,wilson_low,wilson_high,simple_low,simple_high
all reviewed orders,98673,14533,14.73,14.51,14.95,14.51,14.95


**Group size matters.** The same calculation for a small group from P1: orders 8+ days late whose
customer rated *after* the parcel arrived.

In [3]:
small = query("""
    SELECT count(*) AS reviewed, count(*) FILTER (WHERE is_unhappy) AS unhappy
    FROM analytics.fact_order
    WHERE order_status = 'delivered' AND days_late >= 8 AND has_review
      AND review_answer_timestamp >= order_delivered_customer_date
""").iloc[0]
pd.DataFrame([rate_ci(int(kpi["unhappy"]), int(kpi["reviewed"])),
              rate_ci(int(small["unhappy"]), int(small["reviewed"]))],
             index=["all reviewed orders", "8+ days late, rated after arrival"])

,reviewed,unhappy,rate_pct,wilson_low,wilson_high,simple_low,simple_high
all reviewed orders,98673,14533,14.73,14.51,14.95,14.51,14.95
"8+ days late, rated after arrival",76,23,30.26,21.09,41.33,19.93,40.59
